# PHASE 7 – CAMERA GEOMETRY & CALIBRATION

---
### ✅ Module 7.1 – Camera Model
- Pinhole camera (concept via code comments)
- Intrinsic parameters (camera matrix)
- Extrinsic parameters (rotation, translation)
---
### ✅ Module 7.2 – Camera Calibration
- Chessboard-based calibration
- Lens distortion correction
- Undistortion of live camera feed
---
### ✅ Module 7.3 – Stereo Vision
- Depth estimation (formula-based)
- Disparity map (conceptual + practical)
---
### ✅ Mini Projects
- 📏 Distance Measurement (single camera)
- 👁️ Stereo Depth Estimator (simulation-ready)
---
### ⚠️ IMPORTANT PRACTICAL NOTE
Camera calibration REQUIRES a chessboard.
This code is written so that:

- ✔ It guides you step-by-step
- ✔ Does not crash if calibration images are missing
- ✔ Can be reused later with real chessboard images

# ✅ COMPLETE SINGLE FILE CODE (PHASE 7)

In [ ]:
import cv2
import numpy as np
import glob

# =====================================================
# MODULE 7.1 – CAMERA MODEL (PINHOLE MODEL)
# =====================================================
"""
Camera Matrix (Intrinsic Parameters):
| fx  0  cx |
| 0  fy  cy |
| 0   0   1 |

fx, fy → focal length
cx, cy → optical center
"""

# =====================================================
# MODULE 7.2 – CAMERA CALIBRATION (CHESSBOARD METHOD)
# =====================================================

# Chessboard size (number of inner corners)
chessboard_size = (9, 6)

# Prepare object points (0,0,0), (1,0,0), ...
objp = np.zeros((chessboard_size[0] * chessboard_size[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:chessboard_size[0], 0:chessboard_size[1]].T.reshape(-1, 2)

objpoints = []  # 3D points
imgpoints = []  # 2D points

# Load calibration images (optional)
images = glob.glob("calibration_images/*.jpg")

if len(images) == 0:
    print("[INFO] No calibration images found.")
    print("[INFO] Camera calibration skipped.")
    calibrated = False
else:
    for img_path in images:
        img = cv2.imread(img_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        ret, corners = cv2.findChessboardCorners(gray, chessboard_size, None)

        if ret:
            objpoints.append(objp)
            imgpoints.append(corners)

            cv2.drawChessboardCorners(img, chessboard_size, corners, ret)
            cv2.imshow("Calibration", img)
            cv2.waitKey(100)

    cv2.destroyAllWindows()

    ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
        objpoints,
        imgpoints,
        gray.shape[::-1],
        None,
        None
    )

    calibrated = True
    print("[INFO] Camera calibrated successfully")

# =====================================================
# MINI PROJECT 1 – DISTANCE MEASUREMENT (SINGLE CAMERA)
# =====================================================
"""
Distance formula (similar triangles):

Distance = (Known Width * Focal Length) / Pixel Width
"""

KNOWN_WIDTH = 5.0  # cm (example object width)
FOCAL_LENGTH = 700  # approximate (can be refined after calibration)

cap = cv2.VideoCapture(0)

print("Press 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Simple object detection (rectangle)
    edges = cv2.Canny(gray, 100, 200)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > 2000:
            x, y, w, h = cv2.boundingRect(cnt)

            distance = (KNOWN_WIDTH * FOCAL_LENGTH) / w

            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(
                frame,
                f"Distance: {distance:.2f} cm",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

            break

    cv2.imshow("Distance Measurement", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# =====================================================
# MODULE 7.3 – STEREO VISION (CONCEPT + PRACTICE)
# =====================================================
"""
Depth = (Baseline * Focal Length) / Disparity

Baseline → distance between two cameras
"""

# Load stereo images (optional demo)
left_img = cv2.imread("left.jpg", cv2.IMREAD_GRAYSCALE)
right_img = cv2.imread("right.jpg", cv2.IMREAD_GRAYSCALE)

if left_img is None or right_img is None:
    print("[INFO] Stereo images not found.")
    print("[INFO] Stereo depth demo skipped.")
else:
    stereo = cv2.StereoBM_create(numDisparities=16*5, blockSize=15)
    disparity = stereo.compute(left_img, right_img)

    disparity = cv2.normalize(disparity, None, 0, 255, cv2.NORM_MINMAX)
    disparity = np.uint8(disparity)

    cv2.imshow("Disparity Map", disparity)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
